# P24 — Multi-Seed Hardening Pass (canonical 5 seeds {1,2,3,5,42})

**Purpose.** Promote the P24 *thin arms* from single-seed (42 only) to the canonical
5-seed protocol, so their numbers can carry error bars in the thesis instead of being
labelled "indicative". Nothing here invents new science — it is the validation pass
flagged in `Thesis_Final_Structure_and_Remaining_Work.md` §2.

**Non-destructive by design.** This notebook and its two helper scripts
(`p24/seed_report_5.py`, `p24/edge_budget_multiseed.py`) are *new*. No original P24
file is edited or deleted. New seeds land as *new sibling* checkpoint dirs
(`full_nc-accel_only_s1`, ...), exactly as `train_save.py` already names them; the
seed-42 dirs and the original `Run_P24_SO_Compression_Colab.ipynb` are untouched.

**What gets computed**
1. The 4 thin arms at the 4 missing seeds {1,2,3,5}: `accel_only`, `gyro_only`, `aw40`, `aw30`.
2. `full` at seeds {1,2,3,5} — needed *only* to make the RED/GS edge-budget table 5-seed.
3. `no_gz` & `tiny` topped up to seeds {3,5} (they were 3-seed) so every headline P24 number is 5-seed.
4. The two aggregators → `results/p24_seed_report_5.json` and `results/p24_edge_budget_multiseed.json`,
   with paired Wilcoxon tests and GenPerByte + CIs.

**Resumable.** `train_save.py` skips any fold/seed whose checkpoint already exists, so a
Colab disconnect is harmless — just re-run. Seed 42 for every arm already exists and will
be skipped instantly. Estimated GPU time for the *missing* work: ~6–8 h, resumable.


> **Final_PROJECT note (updated 2026-09-26).** All paths now come from
> `Final_PROJECT/project_paths.py` via the first code cell below — the only thing a new user may
> need to edit is the DAGHAR data path in that file (and `FINAL_PROJECT_ROOT` below, only if the
> folder isn't found automatically). **Saved cell outputs are from the original thesis run**, when
> this code lived at `/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression`; they are kept as the record of that run.
> All training scripts are **resumable**: re-running this notebook in place skips any fold whose
> result files already exist, so it will not overwrite the thesis results.

## 1. Mount & paths
Same layout as `Run_P24_SO_Compression_Colab.ipynb`. The frozen ConvNet probs for **all 5**
seeds already exist under P21 `results/sweep/base_s{1,2,3,5,42}` — this notebook only
regenerates the *GNN* side.


In [ ]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("p24")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH
P21_SWEEP    = PP.P21_SWEEP        # per-seed frozen ConvNet base_s* live here
os.environ["P21_SWEEP"] = P21_SWEEP  # used as "$P21_SWEEP" in the shell cells below
assert os.path.isdir(P21_SWEEP), "missing P21_SWEEP"
for s in (1, 2, 3, 5, 42):
    d = os.path.join(P21_SWEEP, f"base_s{s}")
    assert os.path.isdir(d), f"missing {d} — the 5-seed fusion needs every base_s*"

## 2. Setup (deps + data check)


In [ ]:
import setup_colab
setup_colab.install_deps(); assert setup_colab.verify(), "fix DATA_PATH"
# scipy is used by the aggregators for the paired Wilcoxon tests (sign-test fallback otherwise)
import importlib
try:
    import scipy  # noqa
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'])

Installing PyTorch Geometric ...
Done.

Verifying imports ...
  OK torch 2.11.0+cu128  CUDA=True
     GPU Tesla T4
  OK torch_geometric 2.8.0

Checking data at: /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  OK KuHar              3 CSVs
  OK MotionSense        3 CSVs
  OK RealWorld-Thigh    3 CSVs
  OK RealWorld-Waist    3 CSVs
  OK UCI                3 CSVs
  OK WISDM              3 CSVs


## 3. Smoke test (~2 min)
One short arm/seed at 2 epochs to confirm the pipeline, then delete the smoke checkpoint
so the real run retrains it.


In [ ]:
# Smoke test writes to a SEPARATE folder, so the real results/checkpoints/full_nc-accel_only_s1 is never touched.
!python -m p24.train_save --data_path "$DATA_PATH" --targets KuHar --epochs 2 --log_every 1 --nc accel_only --seed 1 --out_root results/_smoke_checkpoints
import torch
ck = torch.load("results/_smoke_checkpoints/full_nc-accel_only_s1/ckpt_KuHar.pt", map_location="cpu", weights_only=False)
print("smoke OK: accel_only params =", ck["n_params"])
import shutil, os
shutil.rmtree("results/_smoke_checkpoints"); print("smoke checkpoints removed")

[P24] train+save | arm=full_nc-accel_only_s1 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=accel_only aw=- | seed=1 | targets=['KuHar'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_nc-accel_only_s1
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 NC: keeping channels [0, 1, 2] -> 3 nodes/graph
[LODO] P21 edge_mode=pearson  select=topk  topk=3  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (1, 2):100%, (0, 2):100%, (0, 1):100%
    [pearson val] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (0, 2):100%, (0, 1):100%, (1, 2):100%
    [pearson test] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (1, 2):100%, (0, 1):100%, (0, 2):100%
    feature_m

---
## 4. Thin arms × 4 missing seeds (the core of the ask)

Each cell = one arm across seeds {1,2,3,5} (seed 42 already exists, skipped automatically).
Cells are independent and resumable — run them in any order, re-run freely after a disconnect.
Per arm ≈ 4 seeds × 6 LODO folds of a small SAGE+DANN model.

> **Tip for parallel wall-clock:** these arms don't collide, so you can open this notebook on a
> second Colab runtime and run different cells there to finish faster. The skip-logic makes it safe.


### 4a. `accel_only` (3-node graph — accelerometer axes only)


In [ ]:
for s in (1, 2, 3, 5):
    !python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc accel_only --seed {s} --epochs 100 --log_every 20

[P24] train+save | arm=full_nc-accel_only_s1 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=accel_only aw=- | seed=1 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_nc-accel_only_s1
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 NC: keeping channels [0, 1, 2] -> 3 nodes/graph
[LODO] P21 edge_mode=pearson  select=topk  topk=3  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (1, 2):100%, (0, 2):100%, (0, 1):100%
    [pearson val] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (0, 2):100%, (0, 1):100%, (1, 2):100%
    [pearson test] edges/window: mean 3.0  min 3  max 3  |

### 4b. `gyro_only` (3-node graph — gyroscope axes only)


In [ ]:
for s in (1, 2, 3, 5):
    !python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc gyro_only --seed {s} --epochs 100 --log_every 20

[P24] train+save | arm=full_nc-gyro_only_s1 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=gyro_only aw=- | seed=1 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_nc-gyro_only_s1
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 NC: keeping channels [3, 4, 5] -> 3 nodes/graph
[LODO] P21 edge_mode=pearson  select=topk  topk=3  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (0, 1):100%, (0, 2):100%, (1, 2):100%
    [pearson val] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (1, 2):100%, (0, 2):100%, (0, 1):100%
    [pearson test] edges/window: mean 3.0  min 3  max 3  | mo

### 4c. `aw40` (Adaptive Windowing → 40 timesteps/window)


In [ ]:
for s in (1, 2, 3, 5):
    !python -m p24.train_save --data_path "$DATA_PATH" --variant full --aw 40 --seed {s} --epochs 100 --log_every 20

[P24] train+save | arm=full_aw40_s1 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=None aw=40 | seed=1 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_aw40_s1
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 AW: 60 -> 40 timesteps/window (strided)
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (0, 1):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (0, 2):100%, (4, 5):100%, (1, 4):100%, (3, 4):100%
    [pearson test] edges/window: mean

### 4d. `aw30` (Adaptive Windowing → 30 timesteps/window)


In [ ]:
for s in (1, 2, 3, 5):
    !python -m p24.train_save --data_path "$DATA_PATH" --variant full --aw 30 --seed {s} --epochs 100 --log_every 20

[P24] train+save | arm=full_aw30_s1 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=None aw=30 | seed=1 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_aw30_s1
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 AW: 60 -> 30 timesteps/window (strided)
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (0, 3):100%, (0, 1):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (3, 4):100%, (4, 5):100%, (1, 2):100%, (2, 5):100%
    [pearson test] edges/window: mean

---
## 5. `full` at the 4 missing seeds — for the 5-seed RED/GS table
The edge-budget study is *inference-only*, but it needs a `full` checkpoint per seed to swap
edge policies on. Seed 42 (`results/checkpoints/full`) already exists and is skipped.


In [ ]:
for s in (1, 2, 3, 5):
    !python -m p24.train_save --data_path "$DATA_PATH" --variant full --seed {s} --epochs 100 --log_every 20

[P24] train+save | arm=full_s1 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=None aw=- | seed=1 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_s1
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (4, 5):100%, (3, 5):100%, (1, 4):100%, (2, 5):100%
    [pearson test] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (1, 2):100

---
## 6. Top up the headline arms `no_gz` & `tiny` to 5 seeds
They already have seeds {42,1,2}; add {3,5} so every headline P24 number uses the same
canonical 5-seed protocol (cheap — no_gz is 5-node, tiny is quarter-width).


In [9]:
for s in (3, 5):
    !python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc no_gz --seed {s} --epochs 100 --log_every 20
    !python -m p24.train_save --data_path "$DATA_PATH" --variant tiny             --seed {s} --epochs 100 --log_every 20

[P24] train+save | arm=full_nc-no_gz_s3 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=no_gz aw=- | seed=3 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_nc-no_gz_s3
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 NC: keeping channels [0, 1, 2, 3, 4] -> 5 nodes/graph
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%, (0, 1):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (1, 4):100%, (0, 1):100%, (3, 4):100%, (0, 3):100%
    [pearson te

---
## 7. Aggregate — 5-seed accuracy report + Wilcoxon + GenPerByte
Fuses each seed's GNN probs with the matching `base_s{seed}` ConvNet (w=0.6), reports
mean±std, per-fold seed-means, GenPerByte with CI, and the paired Wilcoxon (no_gz vs full).
Writes `results/p24_seed_report_5.json`. Safe to run before all seeds finish — arms with
missing seeds simply report a smaller n.


In [10]:
!python -m p24.seed_report_5 --p21_sweep "$P21_SWEEP"


  P24 — CANONICAL 5-SEED report (fused w=0.6, frozen ConvNet per seed)
  seeds requested: [1, 2, 3, 5, 42]   |   canonical headline uses {1,2,3,5,42}
  Reference: P21 winner fused 77.96 ± 0.21% (5-seed)

  arm                   n   fused mean±std   GNN-only   Δ vs full   GenPerByte
  ----------------------------------------------------------------------------
  full (P21 winner)     5   77.96±0.21%    74.41% (reference)       0.0035
  accel_only            5   76.85±0.40%    68.29%     -1.11pp       0.0035
  gyro_only             5   73.14±1.19%    65.79%     -4.81pp       0.0033
  aw40                  5   76.89±0.62%    72.78%     -1.06pp       0.0035
  aw30                  5   75.73±0.22%    70.35%     -2.23pp       0.0034
  no_gz                 5   77.97±0.75%    72.75%     +0.02pp       0.0035
  tiny                  5   76.97±0.74%    72.07%     -0.99pp       0.0205

  Per-fold fused seed-means:
  fold              full (P21 win     accel_only      gyro_only           aw40    

## 8. Aggregate — 5-seed RED/GS edge-budget table
Per-seed inference on the `full_s*` checkpoints, fused against `base_s{seed}`; reports
policy×budget seed mean±std and the paired Wilcoxon for pearson-9 vs RED-9 ("structure ≫ random").
Writes `results/p24_edge_budget_multiseed.json`.


In [11]:
!python -m p24.edge_budget_multiseed --data_path "$DATA_PATH" --p21_sweep "$P21_SWEEP"

  [edge_budget] seed 1 ...
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (4, 5):100%, (3, 5):100%, (1, 4):100%, (2, 5):100%
    [pearson test] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (1, 2):100%, (4, 5):100%, (0, 1):100%, (1, 4):100%
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 18 edges, 90 features each)...
    built 5000/35396 graphs
    built 10000/35396 graphs
    built 15000/35396 graphs
    built 20000/35396 graphs
    built 250

---
## 9. Done — what you now have
- `results/p24_seed_report_5.json` — 5-seed fused/GNN mean±std for all thin + headline arms, per-fold means, GenPerByte+CI, Wilcoxon.
- `results/p24_edge_budget_multiseed.json` — 5-seed RED/GS/Pearson table + pearson-9-vs-RED-9 Wilcoxon.

These promote the P24 arms to "confirmed" and feed directly into thesis **Chapter 7** and
`Thesis_Final_Structure_and_Remaining_Work.md` §2 items 1, 2, 4. The remaining §2 items
(combined-axis Pareto point §2.3, and the novelty-bar finalization §2.5) are
handled elsewhere and don't depend on these runs.


In [12]:
import os, glob
print("cwd:", os.getcwd())
print("PROJECT_PATH:", PROJECT_PATH)
for root in {os.getcwd(), PROJECT_PATH, "/content"}:
    for name in ("p24_seed_report_5.json", "p24_edge_budget_multiseed.json"):
        for h in glob.glob(os.path.join(root, "**", name), recursive=True):
            print("FOUND:", h, os.path.getsize(h), "bytes")
ck = os.path.join(PROJECT_PATH, "results", "checkpoints")
print("checkpoint dirs:", sorted(os.listdir(ck)) if os.path.isdir(ck) else "MISSING")

cwd: /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression
PROJECT_PATH: /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression
FOUND: /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/p24_seed_report_5.json 5066 bytes
FOUND: /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/p24_edge_budget_multiseed.json 5800 bytes
FOUND: /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/p24_seed_report_5.json 5066 bytes
FOUND: /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/p24_edge_budget_multiseed.json 5800 bytes
checkpoint dirs: ['compact', 'full', 'full_aw30', 'full_aw30_s1', 'full_aw30_s2', 'full_aw30_s3', 'full_aw30_s5', 'full_aw40', 'full_aw40_s1', 'full_aw40_s2', 'full_aw40_s3', 'full_aw40_s5', 'full_nc-accel_only', 'full_nc-a